In [0]:
# ===========================================
# Notebook Name:
# 04_build_decks
#
# Purpose:
# Parse raw Pokemon deck HTML and build the
# deck-level Silver table, including the
# functional deck_hash (ADR-001).
#
# Card-level rows are built separately by
# 05_build_deck_cards, since each notebook
# is a distinct task in the Silver job DAG
# and cannot share in-memory state.
#
# Input:
# pokemon.bronze.deck_raw
#
# Output:
# pokemon.silver.decks
#
# Merge key:
# deck_code
#
# Change detection:
# response_hash
# ===========================================

In [0]:
%pip install beautifulsoup4

In [0]:
dbutils.library.restartPython()

In [0]:
import hashlib
from collections import defaultdict
from typing import Any
import re

from bs4 import BeautifulSoup

from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)
from pyspark.sql.window import Window


DECK_RAW_TABLE = "pokemon.bronze.deck_raw"
DECKS_TABLE = "pokemon.silver.decks"

print("Input :", DECK_RAW_TABLE)
print("Output:", DECKS_TABLE)

In [0]:
CATEGORY_NAMES = {
    "ポケモン": "pokemon",
    "グッズ": "goods",
    "ポケモンのどうぐ": "pokemon_tool",
    "サポート": "supporter",
    "スタジアム": "stadium",
    "エネルギー": "energy",
}


def normalize_text(value: str | None) -> str:
    if value is None:
        return ""

    return re.sub(
        r"\s+",
        " ",
        value.replace("\u3000", " "),
    ).strip()


def parse_quantity(value: str) -> int | None:
    normalized = normalize_text(value)

    if re.fullmatch(r"\d+", normalized):
        return int(normalized)

    match = re.fullmatch(
        r"(\d+)\s*枚",
        normalized,
    )

    if match:
        return int(match.group(1))

    return None

In [0]:
def parse_deck_html(
    html: str,
    deck_code: str,
) -> list[dict[str, Any]]:

    if not html:
        return []

    soup = BeautifulSoup(
        html,
        "html.parser",
    )

    cards: list[dict[str, Any]] = []
    current_category: str | None = None

    for row in soup.find_all("tr"):
        row_cells = [
            normalize_text(
                cell.get_text(
                    " ",
                    strip=True,
                )
            )
            for cell in row.find_all(
                ["th", "td"],
                recursive=False,
            )
        ]

        row_cells = [
            cell
            for cell in row_cells
            if cell
        ]

        if not row_cells:
            continue

        first_cell = row_cells[0]

        if first_cell in CATEGORY_NAMES:
            current_category = (
                CATEGORY_NAMES[first_cell]
            )
            continue

        if current_category is None:
            continue

        if any(
            keyword in first_cell
            for keyword in [
                "カード名",
                "小計",
                "合計",
            ]
        ):
            continue

        quantity: int | None = None
        quantity_index: int | None = None

        for index, cell in enumerate(
            row_cells[1:],
            start=1,
        ):
            parsed_quantity = (
                parse_quantity(cell)
            )

            if parsed_quantity is not None:
                quantity = parsed_quantity
                quantity_index = index
                break

        if (
            quantity is None
            or quantity_index is None
        ):
            continue

        remaining_cells = row_cells[
            quantity_index + 1:
        ]

        expansion = (
            remaining_cells[0]
            if len(remaining_cells) >= 1
            else None
        )

        collection_number = (
            remaining_cells[1]
            if len(remaining_cells) >= 2
            else None
        )

        cards.append({
            "deck_code": deck_code,
            "category": current_category,
            "card_name": first_cell,
            "quantity": quantity,
            "expansion": expansion,
            "collection_number": (
                collection_number
            ),
        })

    return cards

In [0]:
def summarize_deck(
    cards: list[dict[str, Any]],
) -> dict[str, Any]:

    category_counts: dict[str, int] = (
        defaultdict(int)
    )

    for card in cards:
        category_counts[
            card["category"]
        ] += int(card["quantity"])

    total_cards = sum(
        int(card["quantity"])
        for card in cards
    )

    return {
        "total_cards": total_cards,
        "card_type_rows": len(cards),
        "unique_card_names": len({
            card["card_name"]
            for card in cards
        }),
        "category_counts": dict(
            category_counts
        ),
        "is_valid_60_cards": (
            total_cards == 60
        ),
    }

In [0]:
def build_deck_hash(
    cards: list[dict[str, Any]],
) -> tuple[str, str]:
    """
    Build a functional deck hash (ADR-001).

    Cards with the same normalized card name
    are aggregated regardless of expansion or
    collection number.
    """
    quantity_by_card_name: dict[
        str, int
    ] = defaultdict(int)

    for card in cards:
        normalized_card_name = (
            normalize_text(
                card.get("card_name")
            )
        )

        if not normalized_card_name:
            raise ValueError(
                "Card name is empty while "
                "generating deck hash"
            )

        quantity_by_card_name[
            normalized_card_name
        ] += int(card["quantity"])

    canonical_lines = [
        f"{card_name}|"
        f"{quantity_by_card_name[card_name]}"
        for card_name in sorted(
            quantity_by_card_name.keys()
        )
    ]

    canonical_string = "\n".join(
        canonical_lines
    )

    deck_hash = hashlib.sha256(
        canonical_string.encode("utf-8")
    ).hexdigest()

    return deck_hash, canonical_string

In [0]:
latest_window = (
    Window
    .partitionBy("deck_code")
    .orderBy(
        F.col("scraped_at").desc(),
        F.col("response_hash").desc(),
    )
)

latest_deck_raw_df = (
    spark.table(DECK_RAW_TABLE)
    .withColumn(
        "_row_number",
        F.row_number().over(
            latest_window
        ),
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

if spark.catalog.tableExists(
    DECKS_TABLE
):
    existing_decks_hash_df = (
        spark.table(DECKS_TABLE)
        .select(
            "deck_code",
            "response_hash",
        )
        .withColumnRenamed(
            "response_hash",
            "existing_response_hash",
        )
    )

else:
    existing_decks_hash_df = (
        spark.createDataFrame(
            [],
            """
            deck_code STRING,
            existing_response_hash STRING
            """,
        )
    )

changed_deck_raw_df = (
    latest_deck_raw_df.alias("raw")
    .join(
        existing_decks_hash_df.alias(
            "existing"
        ),
        on="deck_code",
        how="left",
    )
    .filter(
        F.col(
            "existing_response_hash"
        ).isNull()
        | (
            F.col(
                "existing_response_hash"
            )
            != F.col("response_hash")
        )
    )
    .select("raw.*")
)

print(
    "Latest deck count:",
    latest_deck_raw_df.count(),
)

print(
    "New/changed deck count:",
    changed_deck_raw_df.count(),
)

In [0]:
raw_deck_rows = (
    changed_deck_raw_df
    .select(
        "deck_code",
        "source_url",
        "response_hash",
        "scraped_at",
        "payload",
    )
    .collect()
)

deck_summaries: list[
    dict[str, Any]
] = []
parse_errors: list[
    dict[str, str]
] = []

for index, row in enumerate(
    raw_deck_rows,
    start=1,
):
    deck_code = row["deck_code"]

    print(
        f"[{index}/{len(raw_deck_rows)}] "
        f"Parsing {deck_code}"
    )

    try:
        cards = parse_deck_html(
            html=row["payload"],
            deck_code=deck_code,
        )

        summary = summarize_deck(cards)

        deck_hash, canonical_string = (
            build_deck_hash(cards)
        )

        deck_summaries.append({
            "deck_hash": deck_hash,
            "deck_hash_version": "v1",
            "deck_code": deck_code,
            "source_url": (
                row["source_url"]
            ),
            "response_hash": (
                row["response_hash"]
            ),
            "source_scraped_at": (
                row["scraped_at"]
            ),
            "total_cards": summary[
                "total_cards"
            ],
            "card_type_rows": summary[
                "card_type_rows"
            ],
            "unique_card_names": (
                summary[
                    "unique_card_names"
                ]
            ),
            "is_valid": summary[
                "is_valid_60_cards"
            ],
            "canonical_string": (
                canonical_string
            ),
        })

    except Exception as error:
        parse_errors.append({
            "deck_code": deck_code,
            "error": str(error),
        })

        print("  ERROR:", error)

print(
    "Parsed decks:",
    len(deck_summaries),
)
print(
    "Errors:",
    len(parse_errors),
)

if parse_errors:
    for error in parse_errors:
        print(error)

    raise ValueError(
        f"{len(parse_errors)} deck "
        "parsing errors occurred"
    )

In [0]:
deck_schema = StructType([
    StructField(
        "deck_hash",
        StringType(),
        False,
    ),
    StructField(
        "deck_hash_version",
        StringType(),
        False,
    ),
    StructField(
        "deck_code",
        StringType(),
        False,
    ),
    StructField(
        "source_url",
        StringType(),
        True,
    ),
    StructField(
        "response_hash",
        StringType(),
        True,
    ),
    StructField(
        "source_scraped_at",
        TimestampType(),
        True,
    ),
    StructField(
        "total_cards",
        IntegerType(),
        False,
    ),
    StructField(
        "card_type_rows",
        LongType(),
        False,
    ),
    StructField(
        "unique_card_names",
        LongType(),
        False,
    ),
    StructField(
        "is_valid",
        BooleanType(),
        False,
    ),
    StructField(
        "canonical_string",
        StringType(),
        False,
    ),
])

if deck_summaries:
    decks_df = (
        spark.createDataFrame(
            deck_summaries,
            schema=deck_schema,
        )
        .withColumn(
            "updated_at",
            F.current_timestamp(),
        )
    )

    display(
        decks_df.orderBy("deck_code")
    )

else:
    decks_df = (
        spark.createDataFrame(
            [],
            schema=deck_schema,
        )
        .withColumn(
            "updated_at",
            F.lit(None).cast(
                "timestamp"
            ),
        )
    )

    print(
        "No new/changed decks to "
        "process."
    )

In [0]:
invalid_decks_df = (
    decks_df.filter(
        ~F.col("is_valid")
    )
)

invalid_count = (
    invalid_decks_df.count()
)

if invalid_count > 0:
    display(invalid_decks_df)

    raise ValueError(
        f"{invalid_count} decks do not "
        "contain 60 cards"
    )

print(
    "Validation passed: "
    "all decks contain exactly 60 cards"
)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{DECKS_TABLE}
(
    deck_hash STRING NOT NULL,
    deck_hash_version STRING NOT NULL,
    deck_code STRING NOT NULL,
    source_url STRING,
    response_hash STRING,
    source_scraped_at TIMESTAMP,
    total_cards INT,
    card_type_rows BIGINT,
    unique_card_names BIGINT,
    is_valid BOOLEAN,
    canonical_string STRING,
    updated_at TIMESTAMP
)
USING DELTA
COMMENT
'Normalized Pokemon deck compositions '
'with functional deck hashes'
""")

In [0]:
if deck_summaries:
    decks_df.createOrReplaceTempView(
        "staged_decks"
    )

    spark.sql(f"""
    MERGE INTO {DECKS_TABLE} AS target
    USING staged_decks AS source
    ON target.deck_code = source.deck_code

    WHEN MATCHED
    AND (
        target.response_hash IS NULL
        OR target.response_hash
            <> source.response_hash
    )
    THEN UPDATE SET *

    WHEN NOT MATCHED
    THEN INSERT *
    """)

    print("Silver decks MERGE completed.")

else:
    print(
        "Nothing to merge into "
        f"{DECKS_TABLE}."
    )

In [0]:
display(
    spark.table(DECKS_TABLE)
    .orderBy(
        F.col("updated_at").desc()
    )
    .limit(50)
)